In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

# 支持从项目根目录或 Lesson4 目录启动 notebook。
if not (Path.cwd() / "train_cuda.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from train_cuda import train


class Seq2SeqAttention(nn.Module):
    def __init__(self, n_features, hidden_dim, input_len, target_len, teacher_forcing_prob=0.5):
        super().__init__()
        self.input_len = input_len
        self.target_len = target_len
        self.teacher_forcing_prob = teacher_forcing_prob
        self.encoder = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.decoder = nn.GRU(n_features, hidden_dim, batch_first=True)
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.regression = nn.Linear(2 * hidden_dim, n_features)
        self.scale = hidden_dim ** -0.5

    def forward(self, x):
        source = x[:, :self.input_len]
        target = x[:, self.input_len:]
        encoded, hidden = self.encoder(source)
        keys = self.key(encoded)
        decoder_input = source[:, -1:]
        outputs, weights = [], []

        for i in range(self.target_len):
            query, hidden = self.decoder(decoder_input, hidden)
            scores = torch.bmm(self.query(query), keys.transpose(1, 2)) * self.scale
            alphas = scores.softmax(dim=-1)
            context = torch.bmm(alphas, encoded)
            output = self.regression(torch.cat((context, query), dim=-1))
            outputs.append(output)
            weights.append(alphas.detach())

            if torch.rand(1) < (self.teacher_forcing_prob if self.training else 0):
                decoder_input = target[:, i:i + 1]
            else:
                decoder_input = output

        self.outputs = torch.cat(outputs, dim=1)
        self.alphas = torch.cat(weights, dim=1)
        return self.outputs


def generate_sequences(n=128, variable_len=False, seed=13):
    """生成带噪声的正方形路径及其绕行方向。"""
    corners = np.array([[-1, -1], [-1, 1], [1, 1], [1, -1]])
    rng = np.random.default_rng(seed)
    bases = rng.integers(4, size=n)
    lengths = rng.integers(3, size=n) + 2 if variable_len else [4] * n
    directions = rng.integers(2, size=n)
    points = [
        corners[[(b + i) % 4 for i in range(4)]][::d * 2 - 1][:length]
        + rng.standard_normal((length, 2)) * 0.1
        for b, d, length in zip(bases, directions, lengths)
    ]
    return np.array(points), np.array(directions)


points, directions = generate_sequences(n=256, seed=13)
test_points, test_directions = generate_sequences(seed=19)
full_train = torch.as_tensor(points, dtype=torch.float32)
full_test = torch.as_tensor(test_points, dtype=torch.float32)
target_train, target_test = full_train[:, 2:], full_test[:, 2:]
source_test = full_test[:, :2]

train_data = TensorDataset(full_train, target_train)
test_data = TensorDataset(full_test, target_test)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, generator=torch.Generator())
test_loader = DataLoader(test_data, batch_size=16)

torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Seq2SeqAttention(n_features=2, hidden_dim=2, input_len=2, target_len=2).to(device)
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
train_losses, test_losses = train(model, train_loader, test_loader, loss, optimizer, 100, device)

Epoch 1/100 - loss: 1.0113 - val_loss: 0.9248
Epoch 2/100 - loss: 0.8826 - val_loss: 0.8136
Epoch 3/100 - loss: 0.7608 - val_loss: 0.7112
Epoch 4/100 - loss: 0.6675 - val_loss: 0.6338
Epoch 5/100 - loss: 0.6002 - val_loss: 0.5795
Epoch 6/100 - loss: 0.5467 - val_loss: 0.5411
Epoch 7/100 - loss: 0.5095 - val_loss: 0.5226
Epoch 8/100 - loss: 0.4968 - val_loss: 0.5156
Epoch 9/100 - loss: 0.4777 - val_loss: 0.5106
Epoch 10/100 - loss: 0.4734 - val_loss: 0.5017
Epoch 11/100 - loss: 0.4498 - val_loss: 0.4815
Epoch 12/100 - loss: 0.4409 - val_loss: 0.4511
Epoch 13/100 - loss: 0.4054 - val_loss: 0.4111
Epoch 14/100 - loss: 0.3755 - val_loss: 0.3669
Epoch 15/100 - loss: 0.3416 - val_loss: 0.3363
Epoch 16/100 - loss: 0.3243 - val_loss: 0.3096
Epoch 17/100 - loss: 0.3044 - val_loss: 0.2905
Epoch 18/100 - loss: 0.2905 - val_loss: 0.2699
Epoch 19/100 - loss: 0.2784 - val_loss: 0.2592
Epoch 20/100 - loss: 0.2662 - val_loss: 0.2535
Epoch 21/100 - loss: 0.2582 - val_loss: 0.2410
Epoch 22/100 - loss: 0

In [2]:
gru = nn.GRU(2, 2, batch_first=True)
keys = nn.Linear(2, 2)
x=torch.randn(16, 2, 2)
encoded, hidden = gru(x)

In [3]:
encoded.shape

torch.Size([16, 2, 2])

In [4]:
key = keys(encoded)

In [6]:
key.shape

torch.Size([16, 2, 2])